# Diagnosis Inference Using Medications — DX-Infer Mode

Interactive notebook with **live streaming** that infers the most specific diagnosis
from a base diagnosis plus the patient’s medications, by traversing the IMO Knowledge Graph
via `domainNarrowerByMedications`.

| Component | Detail |
|-----------|--------|
| **LLM** | AWS Bedrock — Claude Haiku (`us.anthropic.claude-haiku-4-5-20251001-v1:0`) |
| **Tools** | `normalize_medical_term`, `get_medication_diagnosis_proto` |
| **Agent** | LangGraph ReAct (`create_react_agent`) |
| **Streaming** | `agent.stream(stream_mode="values")` with live `update_display()` |

## 0. Setup

In [ ]:
import sys, os, re, json, html as html_mod, uuid
from typing import List

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# ── IMO API credentials (remove before pushing to DevOps) ──
os.environ["IMO_CLIENT_ID"]     = ""
os.environ["IMO_CLIENT_SECRET"] = ""
os.environ["IMO_KG_CLIENT_ID"]     = ""
os.environ["IMO_KG_CLIENT_SECRET"] = ""

# ── AWS Bedrock credentials (remove before pushing to DevOps) ──
os.environ.pop("AWS_PROFILE", None)
os.environ["AWS_ACCESS_KEY_ID"]     = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"]     = ""
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

print("✓  Environment configured (dx-infer mode)")

✓  Environment configured (dx-infer mode)


## 1. LLM, Tools & System Prompt

In [9]:
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from kg_api_client import KGApiClient
from typing import Any

llm = ChatBedrockConverse(
    model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    temperature=0,
    provider="anthropic",
    max_tokens=64000,
)
client = KGApiClient()

# --- Token overflow prevention (from route's _cap_tool_response) ---
_KG_LIST_THRESHOLD = 200
_MAX_TOOL_RESPONSE_CHARS = 100000

def _truncate_kg_lists(data, candidate_problems=None):
    if not isinstance(data, dict):
        return data
    list_fields = [
        "treatedProblems", "causedProblems", "domainNarrower", "narrower",
        "synonyms", "associatedTreatments", "contraindicatedTreatments",
        "supportiveTreatments", "domainNarrowerByMedications",
        "domainNarrowerByProcedures", "containingMedications",
    ]
    result = {}
    for key, value in data.items():
        if key in list_fields and isinstance(value, list) and len(value) > _KG_LIST_THRESHOLD:
            total_count = len(value)
            if candidate_problems:
                candidates_lower = [c.lower().strip() for c in candidate_problems]
                matched = [
                    item for item in value
                    if any(
                        cand in item.get("title", "").lower()
                        or item.get("title", "").lower() in cand
                        for cand in candidates_lower
                    )
                ]
                result[key] = matched
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing " + str(len(matched)) + " items matching candidates. "
                    + "Full list too large (" + str(total_count) + " items)."
                )
            else:
                result[key] = value[:_KG_LIST_THRESHOLD]
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing first " + str(_KG_LIST_THRESHOLD) + " of " + str(total_count) + ". "
                    + "Truncated to prevent token overflow."
                )
        elif key == "lexical" and isinstance(value, dict):
            result[key] = _truncate_kg_lists(value, candidate_problems)
        else:
            result[key] = value
    return result

def _aggressive_truncate(data, max_items):
    if not isinstance(data, dict):
        return data
    result = {}
    for key, value in data.items():
        if isinstance(value, list) and len(value) > max_items:
            total = len(value)
            result[key] = value[:max_items]
            result[key + "_summary"] = (
                "KG returned " + str(total) + " total items. "
                + "Showing first " + str(max_items) + " of " + str(total)
                + " (truncated for token limit)."
            )
        elif isinstance(value, dict):
            result[key] = _aggressive_truncate(value, max_items)
        else:
            result[key] = value
    return result

def _cap_tool_response(result, candidate_problems=None):
    result = _truncate_kg_lists(result, candidate_problems)
    output = json.dumps(result, indent=2)
    if len(output) <= _MAX_TOOL_RESPONSE_CHARS:
        return output
    shrink_limit = 50
    while len(output) > _MAX_TOOL_RESPONSE_CHARS and shrink_limit >= 10:
        result = _aggressive_truncate(result, shrink_limit)
        output = json.dumps(result, indent=2)
        shrink_limit = shrink_limit // 2
    if len(output) > _MAX_TOOL_RESPONSE_CHARS:
        result = _aggressive_truncate(result, 5)
        output = json.dumps(result, indent=2)
    return output

# --- Tool definitions (exact match with refine_with_kg route) ---

@tool
def normalize_medical_term(input_term: str, domain: str = "Medication", order_name: str = "") -> str:
    """Normalize a medical term using IMO Precision Normalize API.

    Args:
        input_term: The clinical term (e.g., "Metformin", "diabetes mellitus", "glucose"). Must not be empty.
        domain: "Medication" for drugs, "Problem" for diagnoses, "Lab" for lab results. Default is Medication.
        order_name: Optional lab order name (e.g., "lipid panel", "comprehensive metabolic panel"). Only used when domain="Lab". If you only have the order name and not the component name, pass the order name as input_term instead.
    """
    if not input_term or not input_term.strip():
        return json.dumps({"success": False, "error": "STOP: input_term is empty. Do NOT retry with empty string. Do NOT call this tool again without a valid medication name. Skip and proceed to scoring."})
    result = client.normalize_medical_term(input_term.strip(), domain, order_name=order_name.strip() if order_name else "")
    return json.dumps(result, indent=2)

@tool
def get_medication_diagnosis_proto(diagnosis_code: str, medication_codes: List[str]) -> str:
    """Enter the KG at a base diagnosis lexical, then return only its narrower
    concepts that have ties to the supplied medications. Used to infer a more
    specific diagnosis from a base problem plus a medication list.

    Backed by the domainNarrowerByMedications field on ProblemLexical.

    Results are sorted so non-leaf concepts (numberOfDomainChildren > 0) appear
    first, ordered by descending children count. This ensures drillable candidates
    are visible even when the list is large and truncated.

    Args:
        diagnosis_code: Base diagnosis lexical code from normalize (domain="Problem").
            Example: 334677 for "type 2 diabetes mellitus".
        medication_codes: List of medication lexical codes from normalize
            (domain="Medication"). At least one code is required.
    """
    if not diagnosis_code or not diagnosis_code.strip():
        return json.dumps({"success": False, "error": "STOP: diagnosis_code is empty. Normalize the base diagnosis first (domain=\'Problem\') and pass its default_lexical_code."})
    if not medication_codes:
        return json.dumps({"success": False, "error": "STOP: medication_codes is empty. Normalize each medication (domain=\'Medication\') and pass their default_lexical_codes."})
    result = client.get_medication_diagnosis_proto(diagnosis_code.strip(), medication_codes)

    if result.get("success") and result.get("lexical"):
        items = result["lexical"].get("domainNarrowerByMedications", [])
        if items:
            items.sort(key=lambda x: -(x.get("numberOfDomainChildren") or 0))
            result["lexical"]["domainNarrowerByMedications"] = items

    return _cap_tool_response(result)

DX_INFER_TOOLS = [normalize_medical_term, get_medication_diagnosis_proto]

SYSTEM_PROMPT = """You are a diagnosis-inference agent. Given a clinical note that contains a BASE diagnosis (e.g., "tonsilitis") and a list of medications, you use the IMO Knowledge Graph to infer a MORE SPECIFIC diagnosis. The KG narrows the base diagnosis to only those child concepts that have ties to the given medications.

## WORKFLOW

### Step 1: Extract (NO TOOL CALLS)
From the clinical note, extract:
- The base diagnosis (a single problem term, e.g., "tonsilitis")
- The list of medications
Present a brief summary of what you extracted, then proceed to Step 2.

### Step 2: Normalize the Base Diagnosis
Call normalize_medical_term(input_term=<base diagnosis>, domain="Problem").
Capture its default_lexical_code as the diagnosis_code.

### Step 3: Normalize Each Medication
For each medication in the list, call normalize_medical_term(input_term=<med name>, domain="Medication").
Capture the default_lexical_code of each into a list of medication_codes. Skip any that fail to normalize.

### Step 4: Iteratively Drill Down to the Most Specific Concept

Call get_medication_diagnosis_proto(diagnosis_code=<current code>, medication_codes=[<med codes>]).
The response returns a list of narrower concepts, each with:
- `code` — its lexical code
- `title` — its display title
- `numberOfDomainChildren` — how many further narrower concepts exist below it

You must now iterate. On each round:

1. **Pick the best candidate** from the returned list using ONLY the clinical note’s context (severity, laterality, chronicity, complications, comorbid conditions, etiology, functional class, EF, staging, etc. — whatever the note actually documents). Do not use medical knowledge to invent a match that is not supported by the note.
2. **Inspect `numberOfDomainChildren` on your pick**:
   - If it is `0` (or missing/null), you have reached a leaf. STOP iterating and go to Step 5 with this concept as the inferred diagnosis.
   - If it is `> 0`, this pick still has more specific children in the medication-linked hierarchy. Call get_medication_diagnosis_proto AGAIN with `diagnosis_code=<the pick’s code>` and the SAME `medication_codes` list. Return to sub-step 1 with the new response.
3. **Bounds**: Iterate up to 8 rounds. If the tool returns an empty list at any level, back up and use the last non-empty concept as the final inferred diagnosis.
4. **Ambiguity**: If two candidates match the note equally well and both have children, prefer the one whose title better reflects the strongest clinical signal in the note (e.g., a documented EF, stage, or complication over a generic modifier). If two are equally supported and both are LEAVES, present both.

The tool is backed by the `domainNarrowerByMedications` KG field. If the tool returns an error, report it verbatim and stop — do NOT fabricate results.

### Step 5: Present the Inferred Diagnosis

#### Diagnosis Inference Using Medication

**Base Diagnosis:** (title) (`code`)
**Medications:** (list)

**Inferred Specific Diagnosis:** (final title) (`final code`)

**Refinement Path:**

| Depth | Code | Title | Children Below | Chosen? |
|---|---|---|---|---|
| 0 | (base code) | (base title) | — | starting concept |
| 1 | (code) | (title) | (n) | ✓ picked because … |
| 2 | (code) | (title) | 0 | ✓ leaf — final |

For each row where you had multiple candidates, briefly note WHY you picked that candidate over the others, citing the specific fragment of the clinical note that supports the choice.

If the tool returned an error, state the error verbatim and note that this may indicate the KG service is unavailable or the codes are invalid.

## RULES
- Always normalize the base diagnosis and each medication first — never invent codes.
- ALWAYS pick from the candidates returned by the tool. Never fabricate a concept or invent a code.
- Recurse until `numberOfDomainChildren` is 0. Do not stop early just because a candidate "sounds specific enough" — the KG’s own hierarchy is the source of truth for specificity.
- Never fabricate KG results. If the prototype endpoint returns an error, report it verbatim and stop.
- Do NOT use your own medical knowledge to narrow the diagnosis — only present concepts returned by the KG. You MAY use the note’s own words to decide which returned candidate best fits.
- Before each tool call, briefly explain WHY you’re calling it and which candidate you’re drilling into.

## SUMMARY BLOCK (REQUIRED)
After your final answer, include a summary block wrapped in `[SUMMARY]` and `[/SUMMARY]` tags. This is displayed in a visual explainer panel for non-technical audiences. Write in simple, plain language.

**FORMAT: Exactly 4 paragraphs separated by blank lines.** Each paragraph maps to a visual card:

1. **Paragraph 1 — Clinical Question:** Restate what the user asked in 1-2 plain sentences. Mention the base diagnosis and medications used for inference.
2. **Paragraph 2 — What the Agent Did:** Describe the steps: normalized the diagnosis and medications, queried the Knowledge Graph to narrow the hierarchy using medication associations.
3. **Paragraph 3 — Knowledge Graph Value:** The KG narrowed the base diagnosis hierarchy using medication associations to infer the most specific match. Emphasize that these are **curated, clinically-verified relationships** that cannot be derived from text matching — this is the KG’s unique value.
4. **Paragraph 4 — Recommendation:** State the inferred specific diagnosis and its ICD-10 code.

Keep each paragraph concise (2-4 sentences max). Bold key terms."""

print("✓  LLM, Tools, and System Prompt ready")
print("   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000")
print("   Tools: " + str([t.name for t in DX_INFER_TOOLS]))

✓  LLM, Tools, and System Prompt ready
   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000
   Tools: ['normalize_medical_term', 'get_medication_diagnosis_proto']


## 2. HTML Renderer + Streaming Engine

In [10]:
def esc(text):
    return html_mod.escape(str(text)) if text else ""

AGENT_CSS = """
<style>
.agent-wrap{font-family:system-ui,-apple-system,sans-serif;max-width:900px;margin:0 auto}
.agent-header{background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);color:#fff;padding:18px 24px;border-radius:12px 12px 0 0;text-align:center}
.agent-header h3{margin:0;font-size:1.1rem;font-weight:600}
.agent-header p{margin:4px 0 0;font-size:.85rem;opacity:.85}
.step-box{background:#f8f9fa;border:1px solid #e0e0e0;border-radius:8px;margin:8px 16px;padding:12px 16px}
.step-box.tool-call{background:#e8f0fe;border-color:#1a73e8}
.step-box.tool-result{background:#e6f4ea;border-color:#137333}
.step-box.error{background:#fce8e6;border-color:#c5221f}
.step-label{font-weight:700;font-size:.82rem;margin-bottom:6px}
.step-label.call{color:#1a73e8}
.step-label.result{color:#137333}
.step-label.error{color:#c5221f}
.step-pre{background:rgba(0,0,0,.04);padding:8px;border-radius:4px;font-size:.78rem;overflow-x:auto;white-space:pre-wrap;max-height:300px;overflow-y:auto}
.agent-answer{padding:16px 24px;font-size:.95rem;line-height:1.6}
.agent-answer table{width:100%;border-collapse:collapse;margin:12px 0;font-size:.85rem}
.agent-answer th{background:#f1f3f4;padding:8px 12px;text-align:left;border-bottom:2px solid #dadce0;font-weight:600}
.agent-answer td{padding:8px 12px;border-bottom:1px solid #efefef}
.agent-answer tr:hover td{background:#f8f9fa}
.summary-card{background:linear-gradient(135deg,#e8f5e9,#f1f8e9);border:1px solid #c8e6c9;border-radius:10px;padding:16px 20px;margin:16px 24px}
.summary-card h4{margin:0 0 10px;color:#2e7d32;font-size:.95rem}
.summary-card .card{background:#fff;border-radius:8px;padding:12px 16px;margin:8px 0;box-shadow:0 1px 3px rgba(0,0,0,.08);font-size:.85rem;line-height:1.5}
.summary-card .card b{color:#1b5e20}
.agent-footer{background:#f8f9fa;padding:10px 24px;border-radius:0 0 12px 12px;border-top:1px solid #e0e0e0;font-size:.78rem;color:#666;text-align:center}
.spinner{display:inline-block;width:14px;height:14px;border:2px solid #764ba2;border-top-color:transparent;border-radius:50%;animation:spin .8s linear infinite;margin-right:6px;vertical-align:middle}
@keyframes spin{to{transform:rotate(360deg)}}
</style>
"""

def _render_tool_call(name, args):
    args_json = esc(json.dumps(args, indent=2, default=str))
    return (
        '<div class="step-box tool-call">'
        '<div class="step-label call">\U0001f527 Tool Call: ' + esc(name) + '</div>'
        '<pre class="step-pre">' + args_json + '</pre></div>'
    )

def _render_tool_result(name, content):
    text = str(content) if not isinstance(content, str) else content
    preview = esc(text[:600])
    suffix = "..." if len(text) > 600 else ""
    return (
        '<div class="step-box tool-result">'
        '<div class="step-label result">\u2705 Result: ' + esc(name) + ' (' + str(len(text)) + ' chars)</div>'
        '<pre class="step-pre">' + preview + suffix + '</pre></div>'
    )

def _render_error(msg):
    return (
        '<div class="step-box error">'
        '<div class="step-label error">\u274c Error</div>'
        '<pre class="step-pre">' + esc(msg) + '</pre></div>'
    )

def _parse_summary(text):
    m = re.search(r'\[SUMMARY\](.*?)\[/SUMMARY\]', text, re.DOTALL)
    if not m:
        return None
    paragraphs = [p.strip() for p in m.group(1).strip().split("\n\n") if p.strip()]
    labels = ["Clinical Question", "What the Agent Did", "Knowledge Graph Value", "Recommendation"]
    cards_html = ""
    for i, para in enumerate(paragraphs[:4]):
        label = labels[i] if i < len(labels) else "Card " + str(i + 1)
        cards_html += '<div class="card"><b>' + label + ':</b> ' + esc(para) + '</div>'
    return (
        '<div class="summary-card"><h4>\U0001f4cb Visual Explainer</h4>'
        + cards_html + '</div>'
    )

def _md_to_html(text):
    text = re.sub(r'\[SUMMARY\].*?\[/SUMMARY\]', '', text, flags=re.DOTALL).strip()
    text = re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', text)
    text = re.sub(r'`([^`]+)`', r'<code style="background:#f1f3f4;padding:1px 5px;border-radius:3px;font-size:.85em">\1</code>', text)

    lines = text.split("\n")
    html_parts = []
    in_table = False
    table_rows = []

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("|") and stripped.endswith("|"):
            cols = [c.strip() for c in stripped.strip("|").split("|")]
            if all(re.match(r'^[-:]+$', c) for c in cols):
                continue
            table_rows.append(cols)
            in_table = True
        else:
            if in_table and table_rows:
                html_parts.append(_build_table(table_rows))
                table_rows = []
                in_table = False
            if stripped.startswith("#### "):
                html_parts.append('<h4 style="margin:16px 0 8px;color:#1a237e">' + stripped[5:] + '</h4>')
            elif stripped.startswith("### "):
                html_parts.append('<h4 style="margin:14px 0 6px;color:#333">' + stripped[4:] + '</h4>')
            elif stripped.startswith("- "):
                html_parts.append('<div style="margin:2px 0 2px 16px">\u2022 ' + stripped[2:] + '</div>')
            elif stripped:
                html_parts.append('<p style="margin:6px 0">' + stripped + '</p>')
    if table_rows:
        html_parts.append(_build_table(table_rows))
    return "\n".join(html_parts)

def _build_table(rows):
    if not rows:
        return ""
    ncols = max(len(r) for r in rows)
    header = rows[0] if rows else []
    html = '<div style="overflow-x:auto;margin:12px 0"><table>'
    html += "<thead><tr>"
    for i in range(ncols):
        val = header[i] if i < len(header) else ""
        html += "<th>" + val + "</th>"
    html += "</tr></thead><tbody>"
    for row in rows[1:]:
        html += "<tr>"
        for i in range(ncols):
            val = row[i] if i < len(row) else ""
            html += "<td>" + val + "</td>"
        html += "</tr>"
    html += "</tbody></table></div>"
    return html

def render_agent_run(steps, final_answer="", status="running"):
    parts = [AGENT_CSS, '<div class="agent-wrap">']
    parts.append(
        '<div class="agent-header">'
        '<h3>Diagnosis Inference Agent \u2014 Medication-Guided</h3>'
        '<p>Clinical Note \u2192 Normalize \u2192 domainNarrowerByMedications \u2192 Most Specific Diagnosis</p></div>'
    )
    for step in steps:
        parts.append(step)
    if final_answer:
        summary_html = _parse_summary(final_answer)
        parts.append('<div class="agent-answer">' + _md_to_html(final_answer) + '</div>')
        if summary_html:
            parts.append(summary_html)
    if status == "running":
        parts.append(
            '<div class="agent-footer"><span class="spinner"></span> Agent working\u2026</div>'
        )
    else:
        parts.append('<div class="agent-footer">\u2705 Agent complete</div>')
    parts.append('</div>')
    return "".join(parts)

print("\u2713  HTML Renderer ready (CSS Grid tables, SUMMARY parser)")

✓  HTML Renderer ready (CSS Grid tables, SUMMARY parser)


## 3. Agent + Streaming Runner

In [11]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, DX_INFER_TOOLS, prompt=SYSTEM_PROMPT)

_conversations = {}
_NOTE_MIN_WORDS = 30

def stream_agent(user_message, session_id="notebook", display_handle=None):
    from IPython.display import display, HTML, update_display

    mode = "dx-infer"
    session_key = session_id + "_" + mode
    is_new_clinical_note = len(user_message.split()) >= _NOTE_MIN_WORDS
    if is_new_clinical_note and session_key in _conversations and _conversations[session_key]:
        _conversations[session_key] = []

    history = _conversations.get(session_key, [])
    messages = list(history) + [HumanMessage(content=user_message)]

    steps = []
    final_answer = ""
    handle_id = "dx_infer_" + uuid.uuid4().hex[:6]
    display(HTML(render_agent_run([], "", "running")), display_id=handle_id)

    try:
        for snapshot in agent.stream(
            {"messages": [SystemMessage(content=SYSTEM_PROMPT)] + messages},
            config={"recursion_limit": 60},
            stream_mode="values",
        ):
            last_msg = snapshot["messages"][-1]

            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    steps.append(_render_tool_call(tc["name"], tc.get("args", {})))
                update_display(HTML(render_agent_run(steps, "", "running")), display_id=handle_id)

            elif last_msg.type == "tool":
                steps.append(_render_tool_result(
                    getattr(last_msg, "name", "tool"),
                    last_msg.content,
                ))
                update_display(HTML(render_agent_run(steps, "", "running")), display_id=handle_id)

            elif last_msg.type == "ai" and last_msg.content:
                text = last_msg.content
                if isinstance(text, list):
                    text = "".join(
                        b.get("text", "") for b in text if isinstance(b, dict) and b.get("type") == "text"
                    )
                if text:
                    final_answer = text

        update_display(HTML(render_agent_run(steps, final_answer, "done")), display_id=handle_id)

        _conversations.setdefault(session_key, [])
        _conversations[session_key].append(HumanMessage(content=user_message))
        if final_answer:
            from langchain_core.messages import AIMessage
            _conversations[session_key].append(AIMessage(content=final_answer))
        if len(_conversations[session_key]) > 4:
            _conversations[session_key] = _conversations[session_key][-2:]

    except Exception as exc:
        steps.append(_render_error(str(exc)))
        update_display(HTML(render_agent_run(steps, "", "error")), display_id=handle_id)
        raise

    return final_answer

print("\u2713  Agent created with " + str(len(DX_INFER_TOOLS)) + " tools, streaming runner ready")
print("   recursion_limit=60, conversation history enabled")

✓  Agent created with 2 tools, streaming runner ready
   recursion_limit=60, conversation history enabled


D:\Users\nthonte\AppData\Local\Temp\1\ipykernel_15320\1341503255.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, DX_INFER_TOOLS, prompt=SYSTEM_PROMPT)


## 4. Clinical Scenario — Tonsilitis + Amoxicillin

In [12]:
CLINICAL_NOTE = """Chief Complaint: Sore throat and fever for 3 days.

HPI: 22-year-old female presents with sudden-onset severe sore throat, painful swallowing, and fever to 39.1°C beginning 3 days ago. No cough, no rhinorrhea, no hoarseness. Endorses malaise and mild headache. No prior similar episodes this year. Not immunocompromised, up to date on vaccinations. Recent exposure — roommate diagnosed with strep throat 5 days ago.

PMH: None. Non-smoker.

Medications (started at this visit):
- Amoxicillin 500 mg TID x 10 days

Allergies: NKDA.

Vitals: T 38.7°C, BP 118/72, HR 96, RR 16, SpO2 99%.

Exam: Symmetric tonsillar hypertrophy 3+ bilaterally with white exudate on both tonsils. Uvula midline (no deviation, no peritonsillar bulge or trismus). Tender bilateral anterior cervical lymphadenopathy. No rash. Lungs clear. No hepatosplenomegaly.

Centor Criteria: 4 (fever, tonsillar exudate, tender anterior cervical adenopathy, absence of cough).

Labs:
- Rapid strep antigen test: POSITIVE for group A Streptococcus
- Throat culture pending (confirmatory)
- CBC: WBC 14.2 with neutrophilic predominance
- Monospot: negative

Assessment:
1) Acute streptococcal tonsillitis (group A Streptococcus, GAS-positive rapid antigen), first documented episode — no history of recurrent tonsillitis. Not chronic. No peritonsillar abscess. Bacterial (not viral) etiology.
2) First-line penicillin family antibiotic prescribed given GAS.

Base Diagnosis:
- Tonsilitis

Medications:
- Amoxicillin

Please infer the more specific diagnosis using the medication list."""

print("\u2713  Clinical note loaded (Tonsilitis + Amoxicillin)")
print("   Length: " + str(len(CLINICAL_NOTE)) + " characters")

✓  Clinical note loaded (Tonsilitis + Amoxicillin)
   Length: 1545 characters


## 5. Run Agent — Diagnosis Inference Using Medications (Streaming)

1. **Extract** base diagnosis + medications from the note
2. **Normalize** base diagnosis (Problem) and each medication (Medication)
3. **Iteratively drill** via `get_medication_diagnosis_proto` — pick best candidate, recurse until leaf
4. **Present** inferred diagnosis with refinement path
5. **SUMMARY** block for non-technical audiences

In [ ]:
stream_id = "dxinfer_" + uuid.uuid4().hex[:6]
from IPython.display import display, HTML
display(HTML(render_agent_run([], "", "running")), display_id=stream_id)

result = stream_agent(CLINICAL_NOTE)